# Demo: Post-Download End-to-End Audio Pipeline

This notebook is a **demo harness** for the pipeline after download is already solved.
It uses existing songs from `Raw playlists:tracks` and runs:

1. Build demo metadata
2. Prepare `track_id`-named audio links
3. Extract CLAP embeddings
4. Create train/val/test splits
5. Train audio genre classifier
6. Evaluate on test split

In [1]:
from pathlib import Path
import os
import re
import subprocess
import pandas as pd

from src.data.clean_metadata import clean_metadata
from src.data.make_splits import make_splits

In [2]:
from pathlib import Path
import os
import re
import subprocess
import pandas as pd

from src.data.clean_metadata import clean_metadata
from src.data.make_splits import make_splits

In [3]:
PROJECT_ROOT = Path('.').resolve()
RAW_ROOT = Path('/Users/gabriel/Documents/Spring 2026/Neural Networks/Raw playlists:tracks')
SOURCE_DIRS = [RAW_ROOT / 'R&B', RAW_ROOT / 'Rock']
CATALOG_CSV = PROJECT_ROOT / 'sampled_spotify_tracks.csv'

DEMO_DIR = PROJECT_ROOT / 'data' / 'demo_postdownload'
DEMO_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = DEMO_DIR / 'audio_trackid'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
METADATA_RAW_CSV = DEMO_DIR / 'metadata_raw.csv'
METADATA_CLEAN_CSV = DEMO_DIR / 'metadata_clean.csv'
SPLITS_DIR = DEMO_DIR / 'splits'
EMBED_NPZ = DEMO_DIR / 'audio_embeddings.npz'
MODEL_DIR = PROJECT_ROOT / 'models' / 'demo_postdownload'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, SOURCE_DIRS, DEMO_DIR

(PosixPath('/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier'),
 [PosixPath('/Users/gabriel/Documents/Spring 2026/Neural Networks/Raw playlists:tracks/R&B'),
  PosixPath('/Users/gabriel/Documents/Spring 2026/Neural Networks/Raw playlists:tracks/Rock')],
 PosixPath('/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload'))

In [4]:
def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def parse_artist_title(path: Path):
    stem = path.stem
    if ' - ' in stem:
        artist, title = stem.split(' - ', 1)
    else:
        artist, title = '', stem
    return artist.strip(), title.strip()

def normalize_demo_genre(folder_name: str) -> str:
    f = folder_name.lower().strip()
    if f in {'r&b', 'rnb', 'r and b'}:
        return 'rnb'
    return f

In [5]:
# 1) Inventory already-downloaded songs
audio_exts = {'.mp3', '.flac', '.m4a', '.wav'}
rows = []
for folder in SOURCE_DIRS:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.suffix.lower() in audio_exts:
            artist_guess, title_guess = parse_artist_title(p)
            rows.append({
                'source_folder': folder.name,
                'source_path': str(p.resolve()),
                'artist_guess': artist_guess,
                'title_guess': title_guess,
                'artist_norm': normalize_text(artist_guess),
                'title_norm': normalize_text(title_guess),
                'genre': normalize_demo_genre(folder.name),
            })
inventory = pd.DataFrame(rows)
len(inventory), inventory.head(3)

(75,
   source_folder                                        source_path  \
 0           R&B  /Users/gabriel/Documents/Spring 2026/Neural Ne...   
 1           R&B  /Users/gabriel/Documents/Spring 2026/Neural Ne...   
 2           R&B  /Users/gabriel/Documents/Spring 2026/Neural Ne...   
 
                      artist_guess    title_guess  \
 0                          Amaria           Over   
 1                   Destin Conrad  THE LAST TIME   
 2  Durand Jones & The Indications   Been So Long   
 
                     artist_norm     title_norm genre  
 0                        amaria           over   rnb  
 1                 destin conrad  the last time   rnb  
 2  durand jones the indications   been so long   rnb  )

In [6]:
# 2) Map inventory to catalog when possible, but KEEP all files.
# If no catalog match exists, assign a stable local track_id so both genres remain.
catalog = pd.read_csv(CATALOG_CSV).copy()
catalog['artist_primary'] = catalog['artists'].astype(str).str.split(';').str[0]
catalog['artist_norm'] = catalog['artist_primary'].apply(normalize_text)
catalog['title_norm'] = catalog['track_name'].apply(normalize_text)

needed = [
    'track_id','track_name','artists','album_name','duration_ms','popularity',
    'release_year','artist_norm','title_norm'
]
available = [c for c in needed if c in catalog.columns]

mapped = inventory.merge(
    catalog[available],
    on=['artist_norm','title_norm'],
    how='left',
)

if 'release_year' not in mapped.columns:
    mapped['release_year'] = pd.NA

# Preserve every local file: generate deterministic local IDs when unmatched
mapped['track_id'] = mapped['track_id'].astype(str).replace({'nan': ''})
unmatched = mapped['track_id'].str.len() == 0
mapped.loc[unmatched, 'track_id'] = [f"LOCAL_{i:06d}" for i in range(1, unmatched.sum() + 1)]

# Keep one row per track_id
mapped = mapped.drop_duplicates(subset=['track_id']).reset_index(drop=True)

print('rows by source_folder:', mapped['source_folder'].value_counts().to_dict())
print('rows by genre:', mapped['genre'].value_counts().to_dict())
print('catalog matched:', int((~unmatched).sum()), 'catalog unmatched/local ids:', int(unmatched.sum()))
len(mapped)

rows by source_folder: {'Rock': 50, 'R&B': 25}
rows by genre: {'rock': 50, 'rnb': 25}
catalog matched: 18 catalog unmatched/local ids: 57


75

In [7]:
# 3) Create track_id-named audio links for pipeline compatibility
created = 0
already = 0
for _, row in mapped.iterrows():
    src = Path(row['source_path'])
    tid = str(row['track_id']).strip()
    dst = AUDIO_DIR / f'{tid}.mp3'
    if dst.exists() or dst.is_symlink():
        already += 1
        continue
    os.symlink(src, dst)
    created += 1
{'created_links': created, 'already_present': already, 'audio_dir': str(AUDIO_DIR)}

{'created_links': 0,
 'already_present': 75,
 'audio_dir': '/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload/audio_trackid'}

In [8]:
# 4) Build raw metadata CSV for cleaning
meta = pd.DataFrame({
    'track_id': mapped['track_id'],
    'artist': mapped['artist_guess'].replace('', pd.NA).fillna(
        mapped.get('artists', pd.Series([pd.NA]*len(mapped))).astype(str).str.split(';').str[0]
    ),
    'title': mapped['title_guess'].replace('', pd.NA).fillna(
        mapped.get('track_name', pd.Series([pd.NA]*len(mapped)))
    ),
    'album': mapped.get('album_name', pd.Series([pd.NA]*len(mapped))),
    'genre': mapped['genre'],  # explicit folder labels: rnb / rock
    'duration_ms': mapped.get('duration_ms', pd.Series([pd.NA]*len(mapped))),
    'popularity': mapped.get('popularity', pd.Series([pd.NA]*len(mapped))),
    'release_year': mapped.get('release_year', pd.Series([pd.NA]*len(mapped))),
})
meta = meta.dropna(subset=['genre']).reset_index(drop=True)
meta.to_csv(METADATA_RAW_CSV, index=False)
print('raw metadata genre counts:', meta['genre'].value_counts().to_dict())
len(meta), METADATA_RAW_CSV

raw metadata genre counts: {'rock': 50, 'rnb': 25}


(75,
 PosixPath('/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload/metadata_raw.csv'))

In [15]:
# 5) Clean metadata + make splits
clean_df = clean_metadata(str(METADATA_RAW_CSV), str(METADATA_CLEAN_CSV), min_genre_count=2)
train_df, val_df, test_df = make_splits(str(METADATA_CLEAN_CSV), str(SPLITS_DIR), train_ratio=0.7, val_ratio=0.15, seed=415)
{
    'clean_rows': len(clean_df),
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'test_rows': len(test_df),
    'genres': clean_df['genre'].value_counts().to_dict(),
}

Loaded 75 rows from /Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload/metadata_raw.csv
Dropped 0 rows with missing genre
Dropped 0 rows in genres with <2 samples
Remaining genres (2): ['rnb', 'rock']
Dropped 0 duplicate tracks
Saved 75 tracks (2 genres) -> /Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload/metadata_clean.csv
train:    51 tracks (68.0%)  |  47 artists  |  2 genres
  val:     9 tracks (12.0%)  |  9 artists  |  2 genres
 test:    15 tracks (20.0%)  |  12 artists  |  2 genres


{'clean_rows': 75,
 'train_rows': 51,
 'val_rows': 9,
 'test_rows': 15,
 'genres': {'rock': 50, 'rnb': 25}}

In [16]:
# 6) Extract embeddings from real waveform files
# Prefer CLAP; fallback to librosa embeddings if CLAP deps are missing.
try:
    from src.features.extract_audio import extract_embeddings
    extract_embeddings(
        metadata_csv=str(METADATA_CLEAN_CSV),
        audio_dir=str(AUDIO_DIR),
        out_file=str(EMBED_NPZ),
        batch_size=16,
    )
    embedding_backend = "clap"
except ModuleNotFoundError as e:
    print(f"CLAP unavailable ({e}). Falling back to librosa MFCC-stat embeddings.")
    import numpy as np
    import librosa

    meta_df = pd.read_csv(METADATA_CLEAN_CSV)
    track_ids = []
    embeddings = []
    for _, row in meta_df.iterrows():
        tid = str(row["track_id"])
        path = Path(AUDIO_DIR) / f"{tid}.mp3"
        if not path.exists():
            continue
        y, sr = librosa.load(path, sr=22050, mono=True)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        # 80-dim embedding: 40 means + 40 stds
        emb = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]).astype(np.float32)
        track_ids.append(tid)
        embeddings.append(emb)

    np.savez_compressed(
        EMBED_NPZ,
        track_ids=np.array(track_ids),
        embeddings=np.array(embeddings, dtype=np.float32),
    )
    embedding_backend = "librosa_mfcc"

print({"embedding_file": str(EMBED_NPZ), "backend": embedding_backend})

Resuming: 75 embeddings already saved
Nothing to do (missing_audio=0)
{'embedding_file': '/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/data/demo_postdownload/audio_embeddings.npz', 'backend': 'clap'}


In [17]:
# 7) Train audio model
cmd_train = [
    'python', '-m', 'src.models.train',
    '--modality', 'audio',
    '--classifier', 'logreg',
    '--train_csv', str(SPLITS_DIR / 'train.csv'),
    '--val_csv', str(SPLITS_DIR / 'val.csv'),
    '--embeddings', str(EMBED_NPZ),
    '--out_dir', str(MODEL_DIR),
]
train_run = subprocess.run(cmd_train, capture_output=True, text=True)
print(train_run.stdout)
if train_run.returncode != 0:
    print(train_run.stderr)
train_run.returncode

Audio model: 80-dim embeddings | 2 classes | 51 train / 9 val tracks
  val macro F1: 1.0000
Saved -> /Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/models/demo_postdownload/audio_logreg.joblib



0

In [12]:
from pathlib import Path
import os
import re
import subprocess
import pandas as pd

from src.data.clean_metadata import clean_metadata
from src.data.make_splits import make_splits

In [18]:
# 8) Evaluate audio model on test split
cmd_eval = [
    'python', '-m', 'src.models.evaluate',
    '--modality', 'audio',
    '--classifier', 'logreg',
    '--test_csv', str(SPLITS_DIR / 'test.csv'),
    '--embeddings', str(EMBED_NPZ),
    '--model_dir', str(MODEL_DIR),
]
eval_run = subprocess.run(cmd_eval, capture_output=True, text=True)
print(eval_run.stdout)
if eval_run.returncode != 0:
    print(eval_run.stderr)
eval_run.returncode


audio_logreg  |  test macro F1: 0.9206  (15 tracks)
              precision    recall  f1-score   support

         rnb      1.000     0.800     0.889         5
        rock      0.909     1.000     0.952        10

    accuracy                          0.933        15
   macro avg      0.955     0.900     0.921        15
weighted avg      0.939     0.933     0.931        15

Confusion matrix saved -> /Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/models/demo_postdownload/confusion_audio_logreg.png



0

In [19]:
# 9) Artifact check
artifacts = [
    MODEL_DIR / 'label_encoder.joblib',
    MODEL_DIR / 'audio_logreg.joblib',
    MODEL_DIR / 'confusion_audio_logreg.png',
]
{str(p): p.exists() for p in artifacts}

{'/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/models/demo_postdownload/label_encoder.joblib': True,
 '/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/models/demo_postdownload/audio_logreg.joblib': True,
 '/Users/gabriel/Documents/Spring 2026/Neural Networks/genre-classifier/models/demo_postdownload/confusion_audio_logreg.png': True}